# typedframes + Jupyter notebooks

`typedframes check` reads `.ipynb` files directly -- no conversion step needed.
Errors and warnings are reported as `notebook.ipynb:cell N:line:col`, mapped back
to the cell that actually produced them, rather than a line number in the raw JSON.

Run it from this directory:

```shell
uv run typedframes check ipynb_example.ipynb
```

This notebook demonstrates every diagnostic severity the checker produces --
two warnings and one error -- to show they all work the same in a notebook as
they do in a `.py` file.

In [1]:
from typing import Annotated

import pandas as pd

from typedframes import BaseSchema, Column


class Orders(BaseSchema):
    order_id = Column(type=int)
    customer_id = Column(type=int)
    amount = Column(type=float)


orders: Annotated[pd.DataFrame, Orders] = pd.DataFrame(
    {
        "order_id": [1, 2, 3],
        "customer_id": [10, 20, 30],
        "amount": [9.99, 19.99, 29.99],
    }
)
print(orders)

   order_id  customer_id  amount
0         1           10    9.99
1         2           20   19.99
2         3           30   29.99


IPython magics are tolerated: the checker blanks the magic line in place and keeps
checking the rest of the cell.

In [2]:
%matplotlib inline

total = orders[Orders.amount.s].sum()
print(f"Total revenue: {total:.2f}")

Total revenue: 59.97


## Warnings

`untracked-dataframe` fires when a DataFrame is loaded without `usecols=` or a
schema annotation -- the checker can't see its columns, so it can't validate
anything read from it. It's a warning, not an error: the code isn't wrong, the
checker just has nothing to check. Left unrun since `more_orders.csv` doesn't
actually exist here.

In [ ]:
# ⚠ untracked-dataframe: no usecols=, no schema annotation
more_orders = pd.read_csv("more_orders.csv")

`dropped-unknown-column` fires when `drop(columns=[...])` names a column that
isn't in the DataFrame's known schema -- `orders` only has `order_id`,
`customer_id`, and `amount`. Left unrun since pandas raises `KeyError` for a
`drop()` of a genuinely missing column.

In [ ]:
# ⚠ dropped-unknown-column: "nonexistent" isn't one of Orders' columns
trimmed = orders.drop(columns=["nonexistent"])

## Error

The cell below intentionally accesses a column that doesn't exist (`revenue` --
the real column is `amount`). `typedframes check` catches this at lint time;
actually running the cell would raise a `KeyError` instead, so it's left unrun.

In [ ]:
orders["revenue"]  # real column is "amount" -- typedframes catches this without running anything